In [5]:
import re
from urllib.parse import urlparse

def is_phishing_url(url):
    """
    Flags a URL as potentially suspicious based on common phishing indicators.

    Args:
        url (str): The URL to analyze.

    Returns:
        bool: True if the URL is suspicious, False otherwise.
    """
    print(f"Analyzing URL: {url}")
    suspicious = False
    reasons = []

    if len(url) > 75:
        suspicious = True
        reasons.append(f"URL is excessively long ({len(url)} characters).")

    parsed_url = urlparse(url)
    domain = parsed_url.netloc
    path = parsed_url.path

    ipv4_pattern = re.compile(r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$')
    if ipv4_pattern.match(domain):
        suspicious = True
        reasons.append("Domain is an IP address.")

    if '@' in domain or '%' in domain:
        suspicious = True
        reasons.append("Suspicious character '@' or '%' found in domain.")
    if 'http://' in path or 'https://' in path: # Redirects or misleading parts in path
        suspicious = True
        reasons.append("URL contains 'http://' or 'https://' within the path.")

     phishing_keywords = ['login', 'signin', 'secure', 'verify', 'account', 'paypal', 'bank', 'update', 'webscr', 'confirm']
    for keyword in phishing_keywords:
        if keyword in domain.lower() or keyword in path.lower():
            suspicious = True
            reasons.append(f"Phishing keyword '{keyword}' found in URL.")
            break

    subdomains = domain.split('.')
      clean_subdomains = [s for s in subdomains if s not in ['www', 'com', 'org', 'net', 'co', 'gov', 'edu']]
    if len(clean_subdomains) > 3:
        suspicious = True
        reasons.append(f"Excessive number of subdomains ({len(clean_subdomains)}). This can be a heuristic for phishing.")

    if suspicious:
        print("  -> Suspicious: YES")
        for reason in reasons:
            print(f"     - {reason}")
    else:
        print("  -> Suspicious: NO")
    print("\n")
    return suspicious

print("--- Phishing URL Detection Demonstration ---\n")

phishing_urls = [
    "http://192.168.1.1/login.php?user=admin",
    "http://www.google.com@phishingsite.com/login",
    "https://www.verylongandcomplicateddomainname.example.com.phishingsite.co/secure/login/verify.html?sessionid=1234567890abcdefg",
    "http://bankofamerica.com.secure-login-update.ru/login",
    "https://www.paypal.com-verification.net/webscr?cmd=_login",
    "http://docs.google.com/document/d/1B_hE_uV6mJ_X_yK-vT_p_qZ_s_o_p_q_r_s_t_u_v_w_x_y_z_A_B_C_D_E_F_G_H_I_J_K_L_M_N_O_P_Q_R_S_T_U_V_W_X_Y_Z/edit?usp=sharing", # Long but legitimate
    "https://www.legitimatebank.com/account/details",
    "https://google.com"
]

for url_to_check in phishing_urls:
    is_phishing_url(url_to_check)


--- Phishing URL Detection Demonstration ---

Analyzing URL: http://192.168.1.1/login.php?user=admin
  -> Suspicious: YES
     - Domain is an IP address.
     - Phishing keyword 'login' found in URL.
     - Excessive number of subdomains (4). This can be a heuristic for phishing.


Analyzing URL: http://www.google.com@phishingsite.com/login
  -> Suspicious: YES
     - Suspicious character '@' or '%' found in domain.
     - Phishing keyword 'login' found in URL.


Analyzing URL: https://www.verylongandcomplicateddomainname.example.com.phishingsite.co/secure/login/verify.html?sessionid=1234567890abcdefg
  -> Suspicious: YES
     - URL is excessively long (125 characters).
     - Phishing keyword 'login' found in URL.


Analyzing URL: http://bankofamerica.com.secure-login-update.ru/login
  -> Suspicious: YES
     - Phishing keyword 'login' found in URL.


Analyzing URL: https://www.paypal.com-verification.net/webscr?cmd=_login
  -> Suspicious: YES
     - Phishing keyword 'paypal' found in